# SAM3 Cell Tracking

GT-seeded SAM3 video propagation on CTC sequence 02 (`Fluo-N2DH-SIM+`).

All configuration lives in **Cell 1**. Run cells top-to-bottom.

In [ ]:
import os
import glob
import csv
import datetime
import subprocess
import numpy as np
import tifffile as tif
import cv2
import torch

# ── Paths ──────────────────────────────────────────────────────────────────────
SEQ         = "02"
MINA_ROOT   = "/home/MinaHossain/DMNet_Rina_Tracking/6row_images"
SRC_FRAMES  = f"{MINA_ROOT}/{SEQ}"
GT_SEG_DIR  = f"{MINA_ROOT}/{SEQ}_GT/SEG"
JPEG_DIR    = f"/home/ktracy/sam3-frames/seq{SEQ}"    # full-frame JPEGs for SAM3
OUT_RES     = f"/home/ktracy/sam3-results/{SEQ}_RES"  # CTC output directory
REPO_DIR    = os.path.expanduser("~/ctc-sam3")
RESULTS_CSV = f"{REPO_DIR}/results.csv"

# ── SAM3 checkpoint ────────────────────────────────────────────────────────────
# Run: ls /home/ktracy/sam3-checkpoints/ to confirm the filename.
SAM3_CKPT = "/home/ktracy/sam3-checkpoints/sam3.pt"

# ── Experiment config ──────────────────────────────────────────────────────────
SEED_MODE   = "gt"   # "gt" uses man_seg0000.tif  |  "yolo" raises NotImplementedError
MIN_CELL_PX = 200    # skip mask regions smaller than this (boundary artefacts)

print(f"CUDA: {torch.cuda.is_available()}  -  {torch.cuda.device_count()} GPU(s)")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
print(f"SAM3 ckpt: {SAM3_CKPT}  exists={os.path.exists(SAM3_CKPT)}")

## JPEG frame export

Convert raw float32 TIF frames to uint8 JPEGs that SAM3's video session can read.
Skips frames already exported — safe to re-run after interruption.

In [ ]:
os.makedirs(JPEG_DIR, exist_ok=True)

frame_files = sorted(f for f in os.listdir(SRC_FRAMES) if f.endswith(".tif"))
n_existing  = sum(1 for i in range(len(frame_files))
                  if os.path.exists(f"{JPEG_DIR}/{i:05d}.jpg"))
print(f"JPEG export: {n_existing}/{len(frame_files)} already done, exporting remaining ...")

for i, fname in enumerate(frame_files):
    out_path = f"{JPEG_DIR}/{i:05d}.jpg"
    if os.path.exists(out_path):
        continue
    raw = tif.imread(f"{SRC_FRAMES}/{fname}")
    lo, hi = float(raw.min()), float(raw.max())
    u8 = ((raw - lo) / (hi - lo) * 255).astype(np.uint8) if hi > lo else np.zeros_like(raw, dtype=np.uint8)
    cv2.imwrite(out_path, np.stack([u8] * 3, axis=-1), [cv2.IMWRITE_JPEG_QUALITY, 95])
    if (i + 1) % 20 == 0 or i + 1 == len(frame_files):
        print(f"  {i+1}/{len(frame_files)}")

print(f"Done - {len(os.listdir(JPEG_DIR))} JPEGs in {JPEG_DIR}")

## SAM3 initialization

Loads the predictor and opens a video session on the JPEG directory.

**Triton patch:** `fill_holes_in_mask_scores` calls a Triton connected-components
kernel that overflows CUDA grid limits on frames wider than ~2048 px. The function
only fills cosmetic internal holes in masks and has no effect on tracking quality,
so we replace it with an identity function before the import chain runs.

**Cache warm-up:** `add_tracker_new_points` (the per-instance seeding path) requires
`inference_state["cached_frame_outputs"][frame_idx]` to exist. That cache is populated
by `propagate_in_video`. Running one frame before seeding satisfies this requirement.

In [ ]:
# Patch before importing Sam3VideoPredictor — the reference is captured at import time.
import sam3.model.sam3_video_inference as _svi
_svi.fill_holes_in_mask_scores = lambda mask, **_kw: mask

from sam3.model.sam3_video_predictor import Sam3VideoPredictor

predictor  = Sam3VideoPredictor(checkpoint_path=SAM3_CKPT)
session    = predictor.start_session(resource_path=JPEG_DIR)
session_id = session["session_id"]

jpegs    = sorted(glob.glob(f"{JPEG_DIR}/*.jpg"))
n_frames = len(jpegs)
_probe   = cv2.imread(jpegs[0])
frame_h, frame_w = _probe.shape[:2]
del _probe

print(f"SAM3 session {session_id[:8]}... | {n_frames} frames | {frame_h}x{frame_w}px")

# Warm-up: propagate one frame so cached_frame_outputs[0] exists before seeding.
# add_tracker_new_points() asserts this cache is populated; it cannot seed cold.
print("Warming up frame 0 cache ...")
for _ in predictor.propagate_in_video(
    session_id=session_id,
    propagation_direction="forward",
    start_frame_idx=0,
    max_frame_num_to_track=1,
):
    pass
print("Ready.")

## Frame-0 seeding

Seeds one foreground point per cell at frame 0. SAM3 propagates those instances
forward through the video.

**Why `points=` and not `bounding_boxes=`?**

`add_prompt` has two internal code paths:

- `bounding_boxes=` routes to open-vocabulary detection. It calls `reset_state()`
  on every invocation, wiping all prior seeds, and ignores `obj_id` entirely.
- `points=` + `obj_id=` routes to `add_tracker_new_points()`. State accumulates
  across calls, each object gets its own ID, and propagation uses instance-tracking
  mode instead of open-vocabulary detection.

Always use the `points=` path for named-instance seeding.

In [ ]:
def seed_from_mask_centroid(frame_idx, obj_id, binary_mask):
    ys, xs = np.where(binary_mask)
    predictor.add_prompt(
        session_id=session_id,
        frame_idx=frame_idx,
        points=[[float(xs.mean()) / frame_w, float(ys.mean()) / frame_h]],
        point_labels=[1],
        obj_id=obj_id,
    )

if SEED_MODE == "gt":
    seg0   = tif.imread(f"{GT_SEG_DIR}/man_seg0000.tif")
    seeded = 0
    for obj_id in np.unique(seg0):
        if obj_id == 0:
            continue
        binary = seg0 == obj_id
        if binary.sum() < MIN_CELL_PX:
            continue
        seed_from_mask_centroid(0, int(obj_id), binary)
        seeded += 1
    print(f"Seeded {seeded} cells from GT man_seg0000.tif")
else:
    raise NotImplementedError(
        "YOLO seeding not yet implemented.\n"
        "Steps: run YOLO on each strip of frame 0, map strip coords to full-frame Y, "
        "then call seed_from_mask_centroid() with each detection centroid."
    )

## Propagation

`propagate_in_video` yields one dict per frame:

```
{
  "frame_index": int,
  "outputs": {
      "out_obj_ids":      np.ndarray  # shape (n,)    - the obj_ids we seeded
      "out_binary_masks": np.ndarray  # shape (n,H,W) - bool, already thresholded
  }
}
```

`outputs` is `None` for frames where no tracked objects are visible.

In [ ]:
masks     = {}   # frame_idx -> (H, W) uint16 label array
track_log = {}   # obj_id -> {"start": int, "end": int}

for result in predictor.propagate_in_video(
    session_id=session_id,
    propagation_direction="forward",
    start_frame_idx=0,
    max_frame_num_to_track=None,
):
    fi      = result["frame_index"]
    outputs = result["outputs"]
    lab     = np.zeros((frame_h, frame_w), dtype=np.uint16)

    if outputs is not None:
        for obj_id, mask in zip(outputs["out_obj_ids"], outputs["out_binary_masks"]):
            oid = int(obj_id)
            lab[mask] = oid
            if oid not in track_log:
                track_log[oid] = {"start": fi, "end": fi}
            else:
                track_log[oid]["end"] = fi

    masks[fi] = lab

    if (fi + 1) % 50 == 0 or fi + 1 == n_frames:
        n_active = len(outputs["out_obj_ids"]) if outputs else 0
        print(f"  {fi+1:4d}/{n_frames}  active cells: {n_active}")

print(f"\nDone - {len(masks)} frames, {len(track_log)} tracks")

## CTC output

Writes `mask####.tif` (uint16 label array) and `res_track.txt` in CTC format.

Gap-fills brief tracking lapses by forward-filling the previous frame's pixels,
keeping each track contiguous as required by the CTC evaluation software.

In [ ]:
import shutil

if os.path.exists(OUT_RES):
    shutil.rmtree(OUT_RES)
os.makedirs(OUT_RES)

# Build per-cell frame lists from the propagation output
cell_frames = {}
for fi, lab in sorted(masks.items()):
    for obj_id in np.unique(lab):
        if obj_id == 0:
            continue
        cell_frames.setdefault(int(obj_id), []).append(fi)

# Forward-fill gaps so each track is contiguous (CTC requirement)
gaps_filled = 0
for obj_id, frames in cell_frames.items():
    frame_set = set(frames)
    prev_lab  = None
    for f in range(frames[0], frames[-1] + 1):
        if f in frame_set:
            prev_lab = masks[f]
        elif prev_lab is not None:
            src_px = prev_lab == obj_id
            if src_px.any():
                masks[f][src_px] = obj_id
                gaps_filled += 1

# Write mask TIFFs
for fi, lab in sorted(masks.items()):
    tif.imwrite(f"{OUT_RES}/mask{fi:04d}.tif", lab.astype(np.uint16))

# Write res_track.txt  (format: obj_id  birth_frame  death_frame  parent_id)
track_spans = {oid: (frames[0], frames[-1]) for oid, frames in cell_frames.items()}
with open(f"{OUT_RES}/res_track.txt", "w") as f:
    for oid, (start, end) in sorted(track_spans.items()):
        f.write(f"{oid} {start} {end} 0\n")

print(f"Wrote {len(masks)} mask TIFFs to {OUT_RES}")
print(f"res_track.txt: {len(track_spans)} tracks")
if gaps_filled:
    print(f"Gap-filled {gaps_filled} frames")
durations = [e - s + 1 for s, e in track_spans.values()]
print(f"Track lengths: min={min(durations)}  median={int(np.median(durations))}  max={max(durations)}")

## Evaluation and results logging

Computes SEGMeasure (Jaccard overlap, pure Python) and TRAMeasure (via traccuracy),
then appends a row to `results.csv` and commits it.

**Baselines to beat:** SEG = 0.422, TRA = 0.525 (DMNet HP4_TRA)

In [ ]:
# ── SEGMeasure (Jaccard) ──────────────────────────────────────────────────────
def compute_seg(gt_seg_dir, res_dir, n_digits=4):
    total_j, total_cells = 0.0, 0
    for fname in sorted(f for f in os.listdir(gt_seg_dir) if f.startswith("man_seg")):
        frame_num = int(fname.replace("man_seg", "").replace(".tif", ""))
        gt        = tif.imread(f"{gt_seg_dir}/{fname}")
        res_path  = f"{res_dir}/mask{frame_num:0{n_digits}d}.tif"
        res       = tif.imread(res_path) if os.path.exists(res_path) else np.zeros_like(gt)
        for gt_id in np.unique(gt):
            if gt_id == 0:
                continue
            gt_cell = gt == gt_id
            best_j  = max(
                (
                    np.logical_and(gt_cell, res == res_id).sum() /
                    np.logical_or( gt_cell, res == res_id).sum()
                    for res_id in np.unique(res) if res_id != 0
                    if np.logical_and(gt_cell, res == res_id).sum() > 0
                ),
                default=0.0,
            )
            total_j     += best_j
            total_cells += 1
    return total_j / total_cells if total_cells else 0.0

seg = compute_seg(GT_SEG_DIR, OUT_RES)
print(f"SEG = {seg:.4f}  (baseline 0.422)")

# ── TRAMeasure ────────────────────────────────────────────────────────────────
tra = det = lnk = None
try:
    from traccuracy.loaders import load_ctc_data
    from traccuracy.matchers import CTCMatcher
    from traccuracy.metrics import CTCMetrics

    gt_tra_dir = f"{MINA_ROOT}/{SEQ}_GT/TRA"
    gt_graph   = load_ctc_data(gt_tra_dir, track_path=f"{gt_tra_dir}/man_track.txt", run_checks=False)
    pred_graph = load_ctc_data(OUT_RES,    track_path=f"{OUT_RES}/res_track.txt",    run_checks=False)
    matched    = CTCMatcher().compute_mapping(gt_graph, pred_graph)
    results    = CTCMetrics().compute(matched)
    tra = results.results["TRA"]
    det = results.results["DET"]
    lnk = results.results["LNK"]
    print(f"TRA = {tra:.4f}  DET = {det:.4f}  LNK = {lnk:.4f}  (baseline TRA 0.525)")
except Exception:
    import traceback as _tb
    _tb.print_exc()

# ── results.csv ───────────────────────────────────────────────────────────────
RUN_NAME = f"sam3_{SEED_MODE}_{SEQ}"

new_row = {
    "run_name":  RUN_NAME,
    "date":      datetime.date.today().isoformat(),
    "seq":       SEQ,
    "seed_mode": SEED_MODE,
    "SEG":       f"{seg:.4f}",
    "TRA":       f"{tra:.4f}" if tra is not None else "",
    "DET":       f"{det:.4f}" if det is not None else "",
    "LNK":       f"{lnk:.4f}" if lnk is not None else "",
}

rows, updated, fieldnames = [], False, None
if os.path.exists(RESULTS_CSV):
    with open(RESULTS_CSV, newline="") as f:
        reader     = csv.DictReader(f)
        fieldnames = list(reader.fieldnames)
        for row in reader:
            rows.append(new_row if row["run_name"] == RUN_NAME else row)
            if row["run_name"] == RUN_NAME:
                updated = True
if not fieldnames:
    fieldnames = list(new_row.keys())
if not updated:
    rows.append(new_row)

with open(RESULTS_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

action = "Updated" if updated else "Appended"
print(f"\n{action} '{RUN_NAME}' in results.csv")

try:
    subprocess.run(["git", "-C", REPO_DIR, "add", "results.csv"], check=True)
    subprocess.run(
        ["git", "-C", REPO_DIR, "commit", "-m",
         f"results: {RUN_NAME} SEG={seg:.4f} TRA={f'{tra:.4f}' if tra else 'n/a'}"],
        check=True,
    )
    print("results.csv committed")
except subprocess.CalledProcessError:
    print("git commit skipped (nothing new to commit)")